# NGC 1068 fluctuated Case 1: thermal CPL

Fits the 3- and 24-month fluctuated representative files, saves both global-fit results, and produces separate representative-realization and 300-seed-median SED plots.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import display

UTILS_DIR = Path(
    "/Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/"
    "continuum_fit/AGN/Fluctuate_True"
)
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

from agn_cosi_fit_utils import save_agn_fit_summary
from agn_ngc1068_fit import (
    build_sed_products,
    evaluate_flux_curves,
    fit_manifest_spectrum,
    plot_sed,
)
%matplotlib inline


In [ ]:
BACKGROUND_PSEUDOCOUNT = None  # np.finfo(float).tiny inside the SED likelihood
SED_ENSEMBLE_RECOMPUTE = False
N_SED_BINS = 10

manifest_paths = {
    3: Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/GammaRay/Paper_Models/Fluctuate_True/Sensitivity_Ensemble/NGC1068_Case1/NGC1068_Case1_CPL_median_realization_3months.json"),
    24: Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/GammaRay/Paper_Models/Fluctuate_True/Sensitivity_Ensemble/NGC1068_Case1/NGC1068_Case1_CPL_median_realization_24months.json"),
}
FIT_SUMMARY_PATH = Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/Papers/AGN_Corona_EC_Overleaf/Fits/Fluctuate_True/NGC1068_Case1_CPL_128_24Months_fit_summary_3_24Months.txt")
PLOT_DIR = Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/Papers/AGN_Corona_EC_Overleaf/Plots/Fluctuate_True")


In [ ]:
fits = {}
for exposure_months, manifest_path in manifest_paths.items():
    fits[exposure_months] = fit_manifest_spectrum(
        manifest_path,
        dataset_name=f"ngc1068_case1_{exposure_months}m",
        background_pseudocount=BACKGROUND_PSEUDOCOUNT,
    )
    bundle = fits[exposure_months]
    print(f"{exposure_months} months: source TS = {bundle['source_ts']:.3f}")
    if bundle["added_component_delta_ts"] is not None:
        print(
            f"{exposure_months} months: added-component Delta TS = "
            f"{bundle['added_component_delta_ts']:.3f}"
        )
    display(bundle["likelihood"].results.get_data_frame())


In [ ]:
fit_labels = {exposure: f"{exposure}m_case1" for exposure in fits}
fit_summary_path = save_agn_fit_summary(
    output_path=FIT_SUMMARY_PATH,
    fit_results={fit_labels[e]: fits[e]["likelihood"].results for e in fits},
    injected_models={fit_labels[e]: {"spectrum": fits[e]["injected_shape"]} for e in fits},
    ts_values={fit_labels[e]: fits[e]["source_ts"] for e in fits},
    exposure_months={fit_labels[e]: e for e in fits},
    extra_statistics={
        fit_labels[e]: {"Delta_TS_added_component_vs_CPL": fits[e]["added_component_delta_ts"]}
        for e in fits if fits[e]["added_component_delta_ts"] is not None
    },
    notes=(
        "Fluctuated NGC 1068 representative files. Exact zero-source null; "
        "background pseudocount is np.finfo(float).tiny when configured as None."
    ),
)
print(f"Saved: {fit_summary_path}")


In [ ]:
fit_24m = fits[24]
flux_curves_24m = evaluate_flux_curves(fit_24m)
sed_products_24m = build_sed_products(
    fit_24m,
    n_sed_bins=N_SED_BINS,
    background_pseudocount=BACKGROUND_PSEUDOCOUNT,
    recompute_ensemble=SED_ENSEMBLE_RECOMPUTE,
    expected_seed_count=300,
)
display(sed_products_24m["representative"][[
    "bin_index", "ts_value", "plot_role", "excess_counts",
]])
display(sed_products_24m["ensemble_summary"][[
    "bin_index", "ts_median", "plot_role", "fraction_ts_zero",
]])


## Plot A — representative median data set

SED values and classification use only the selected representative realization’s bin likelihood and TS.


In [ ]:
representative_plot_path = PLOT_DIR / "NGC1068_Case1_CPL_128_24Months_SED_RepresentativeMedianDataset.pdf"
fig_representative, ax_representative = plot_sed(
    fit_24m,
    flux_curves_24m,
    sed_products_24m["representative"],
    title="NGC 1068\nThermal (Case 1)\n24-month",
    color="#D55E00",
    save_path=representative_plot_path,
)
plt.show()


## Plot B — 300-seed median SED

SED values, intervals, and classification use the 300-seed bin-by-bin ensemble medians.


In [ ]:
ensemble_plot_path = PLOT_DIR / "NGC1068_Case1_CPL_128_24Months_SED_300SeedMedianSED.pdf"
fig_ensemble, ax_ensemble = plot_sed(
    fit_24m,
    flux_curves_24m,
    sed_products_24m["ensemble"],
    title="NGC 1068\nThermal (Case 1)\n24-month",
    color="#D55E00",
    save_path=ensemble_plot_path,
)
plt.show()
